In [ ]:
# Imports goes here.

import torch
from datasets import load_dataset
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer
)
import json

I am taking small subset of the dataset(hindi) instead of whole, since the whole dataset size is around ~33gb which is huge.

In [7]:
from huggingface_hub import notebook_login
notebook_login()

In [8]:
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicVoices",
    "hindi",
    split="train",
    streaming=True
)

dataset = list(dataset.take(50))

DatasetNotFoundError: Dataset 'ai4bharat/IndicVoices' is a gated dataset on the Hub. You must be authenticated to access it.

I am building tokensizer (from the small dataset we took ie frist 50)

In [ ]:
texts = [x["text"] for x in dataset]

vocab = set()
for text in texts:
    for char in text:
        if char == " " or "\u0900" <= char <= "\u097F":
            vocab.add(char)

vocab = sorted(list(vocab))

vocab_dict = {char: i for i, char in enumerate(vocab)}
vocab_dict["[PAD]"] = len(vocab_dict)
vocab_dict["[UNK]"] = len(vocab_dict)

with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False)

The below cell will create a processor which we imported earlier.
Note: I am considering sampling rate as 16kHz which is standard.

In [ ]:
tokenizer = Wav2Vec2CTCTokenizer(
    "vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token=" "
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

I have loaded facebook/wav2vec-base which is quite SMALL model, since I have trained it in my local machine. I would try to train with wav2vec2-large-xlsr-300m, if I have sufficient GPU's in future.

In [1]:
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-base",
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ignore_mismatched_sizes=True
)

NameError: name 'Wav2Vec2ForCTC' is not defined

In [ ]:
model.to("cuda")

In [ ]:
# These lines are for memory optimizations (which are CRITICAL for my NVIDIA GPU to train).

model.freeze_feature_encoder()
model.gradient_checkpointing_enable()

In [ ]:
def prepare_batch(batch):
    audio = batch["audio_filepath"]

    audio_array = audio["array"]
    sr = audio["sampling_rate"]

    # Convert stereo → mono
    if len(audio_array.shape) > 1:
        audio_array = audio_array.mean(axis=1)

    # Limit audio length (for 5 sec)
    if len(audio_array) > 16000 * 5:
        audio_array = audio_array[:16000 * 5]

    # if sample rate isn't 16kHz, we will make it 16kHz.
    if sr != 16000:
        import torchaudio
        audio_array = torch.tensor(audio_array, dtype=torch.float32)
        resampler = torchaudio.transforms.Resample(sr, 16000)
        audio_array = resampler(audio_array).numpy()

    batch["input_values"] = processor(
        audio_array,
        sampling_rate=16000
    ).input_values[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

In [ ]:
dataset = [prepare_batch(x) for x in dataset]

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Union

@dataclass
class DataCollatorCTCWithPadding:
    processor: any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_values = [f["input_values"] for f in features]
        labels = [f["labels"] for f in features]

        batch = self.processor.feature_extractor.pad(
            {"input_values": input_values},
            return_tensors="pt"
        )

        labels_batch = self.processor.tokenizer.pad(
            {"input_ids": labels},
            return_tensors="pt",
            padding=True
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        batch["labels"] = labels
        return batch

In [ ]:
training_args = TrainingArguments(
    output_dir="./wav2vec2-indic",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    learning_rate=1e-4,
    fp16=False,
    logging_steps=1,
    save_steps=10
)

In [ ]:
data_collator = DataCollatorCTCWithPadding(processor=processor)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

In [ ]:
trainer.train()